## Predictive Maintenance Dashboard

Loads sensor data, trains a Ridge regression model, and provides interactive forecasts for 5 parameters or overall sensor health. Includes time‑series plots and a web interface for manual prediction requests.

## Environment Setup

This cell initializes the Python environment for sensor data analysis and predictive maintenance. It loads libraries for data processing, machine learning, interactive visualizations, and the dashboard interface.

In [1]:
# Date handling and random generation
import datetime
import random as ra

# Data processing and analysis
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt

# ML models and cross-validation
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.model_selection import RepeatedKFold, cross_val_score

# Interactive charts
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Dashboard interface
from dash import Dash, dcc, html, Output, Input, State

## Core Functions

This cell contains the main logic for prediction, visualization, and dashboard callbacks. It includes:

- Ridge regression training and evaluation.
- Parameter‑wise and aggregate sensor health predictions.
- Date formatting and interactive chart generation.

In [2]:
# Counts how many 5‑record blocks remain within normal range
def predict_ridge_model(model, size, y, scope):
    predicted_x, begin, end = [], scope[0], scope[1]
    size -= size % 5  # round down to a multiple of 5
    for i in range(0, size, 5):
        x = model.predict([y[i:i + 5]])[0]
        predicted_x.append(x)
        if x < begin or end < x:
            break
    else:
        return f'{size}+'
    return len(predicted_x)


# Trains Ridge regression, returns parameter‑specific prediction and MAE
def forming_characteristics(data, column, scope):
    data = data[parameters]
    X, y = data.values, data[column].values
    model = Ridge(alpha=1.0)
    cv = RepeatedKFold(n_splits=10, n_repeats=3, random_state=100)
    scores = cross_val_score(model, X, y, scoring='neg_mean_absolute_error', cv=cv, n_jobs=-1)
    scores = np.absolute(scores)
    model.fit(X, y)
    predicted_x = predict_ridge_model(model, len(data), y, scope)
    print(f'{column}: {predicted_x}')
    print(f'Mean MAE: {np.mean(scores)} ({np.std(scores)})')
    return predicted_x


# Weighted sum of all parameter predictions
def general_predict(predict_days):
    summary = 0
    for column in parameters:
        weight, days = weights[column], predict_days[column]
        try:
            summary += weight * days
        except TypeError:   # days is a string like "200+"
            days = int(days[:-1])
            summary += weight ** 2 * days
    return round(summary)


# Runs forming_characteristics for all parameters on a data slice
def predict_parameters(begin, end):
    for column in parameters:
        predict_days[column] = forming_characteristics(data[begin:end], column, scope=scopes[column])
    return predict_days


# Russian pluralisation: returns '1 дня' or 'N дней'
def form_days(day):
    if day == 1:
        return '1 дня'
    return f'{day} дней'


# Adds days to a date and returns a DD.MM.YYYY string
def rebuild_date(starting_date, days):
    return (starting_date + datetime.timedelta(days=days)).strftime("%d.%m.%Y")


# Builds interactive plots for 5 parameters over a time slice
def create_figure(data, begin, end):
    data = data[begin:end]
    fig = make_subplots(rows=5, cols=1, vertical_spacing=0.1, subplot_titles=titles)
    for column, i in zip(parameters, range(1, 6)):
        fig.add_trace(px.line(data, x=data.time, y=column)['data'][0], row=i, col=1)
        fig.update_traces(line_color=ra.choice(colors), row=i, col=1)
    fig.update_layout(autosize=False, width=1800, height=1200)
    return fig

## Data Loading & Configuration

Loads the dataset, color palette, and dictionaries for parameter names and normal ranges.  
Only the first 30 records are used for quick preview.

In [3]:
# Load palette, sample data, and parameter dictionaries
with open('../data/colors.txt') as file:
    colors = file.read().replace('\n', ' ').split(', ')

amount = 30
titles = ['Динамический диапазон', 'Угол обзора камеры', 'Фокусное расстояние', 'Температура', 'Частота колебаний']
data = pd.read_excel('../data/database.xlsx')[:amount]
parameters = list(data.columns[1:])
transcripts = {column: title for column, title in zip(parameters, titles)}
scopes = {columns: ranges for columns, ranges in zip(parameters,
                                               [(4, 9), (6, 160), (2.8, 16), (-10, 60), (8, 120)])}

display(data.head(10))
scopes

,time,dynamic_range,viewing_angle,focal_length,temperature,oscillation_frequency
0,16.07.2009,7,15,3.395921,8,41
1,17.07.2009,4,88,7.910634,22,84
2,18.07.2009,4,79,7.070259,41,95
3,19.07.2009,5,33,6.621067,54,69
4,20.07.2009,6,90,12.573989,2,55
5,21.07.2009,5,84,3.460187,16,45
6,22.07.2009,7,56,12.495441,44,78
7,23.07.2009,6,149,12.323401,29,44
8,24.07.2009,4,43,8.174660,39,51
9,25.07.2009,7,71,4.940332,38,90


{'dynamic_range': (4, 9),
 'viewing_angle': (6, 160),
 'focal_length': (2.8, 16),
 'temperature': (-10, 60),
 'oscillation_frequency': (8, 120)}

## Correlation & Feature Weights

Computes the correlation matrix between parameters and derives weights for the weighted prediction. The weights reflect each parameter's average absolute correlation with the others.

In [4]:
# Correlation matrix and feature weights for weighted prediction
corr = data.corr(numeric_only=True)
display(corr.style.background_gradient(cmap='coolwarm'))

weights = (corr.apply(abs).reset_index(drop=True).sum() - 1) / 4
weights

,dynamic_range,viewing_angle,focal_length,temperature,oscillation_frequency
dynamic_range,1.000000,0.028981,0.349042,0.166618,-0.118519
viewing_angle,0.028981,1.000000,0.217009,-0.060653,0.131033
focal_length,0.349042,0.217009,1.000000,0.232661,-0.425236
temperature,0.166618,-0.060653,0.232661,1.000000,-0.008566
oscillation_frequency,-0.118519,0.131033,-0.425236,-0.008566,1.000000


dynamic_range            0.165790
viewing_angle            0.109419
focal_length             0.305987
temperature              0.117124
oscillation_frequency    0.170838
dtype: float64

## Interactive Dashboard

Launches the Dash web interface with:
- Interactive time‑series plots for 5 parameters.
- Input field to request a prediction for a specific parameter (1–5) or a general health forecast ("General").

In [5]:
# Use first 200 records for training (begin=10 skips initial transient)
external_stylesheets = ['https://codepen.io/chriddyp/pen/bWLwgP.css']
app = Dash(__name__, external_stylesheets=external_stylesheets)
begin, end, predict_days = 10, 200, {}
starting_date = datetime.datetime.strptime(data.time[begin], "%d.%m.%Y")
predict_days = predict_parameters(begin, end)

app.layout = html.Div([
    html.H2('FZ-SC2M 374930'),
    dcc.Graph(id='main-graph', figure=create_figure(data, begin, end)),
    html.Div(id='description', children='Введите номер параметра'),
    html.Div(dcc.Input(id='input-on-submit', type='text')),
    html.Button('Выполнить', id='submit-val', n_clicks=0),
    html.Div(id='container-button')
])

@app.callback(
    Output('container-button', 'children'),
    Input('submit-val', 'n_clicks'),
    State('input-on-submit', 'value')
)
def update_output(n_clicks, value):
    if not n_clicks:
        return None

    value = value.strip()

    # User wants overall sensor health (weighted sum)
    if value.lower() == 'общий':
        days = general_predict(predict_days)
        return f'Проведите технический осмотр датчика в течение {form_days(days)} (до {rebuild_date(starting_date, days)})'

    # Validate that input is exactly a digit from 1 to 5
    if value not in ('1', '2', '3', '4', '5'):
        return 'Введите число от 1 до 5 или слово "Общий" для получения прогноза по всем параметрам'

    # Specific parameter forecast
    column = parameters[int(value) - 1]
    days = predict_days[column]

    # If the model gave "200+" it means it never exceeded limits in the whole sample
    if type(days) == str:
        days = int(days[:-1])          # remove trailing '+'
        date = rebuild_date(starting_date, days)
        return f'В течение {form_days(days)} (до {date}) параметр "{transcripts[column]}" будет находиться в пределах нормы'

    # Otherwise it's a precise number of days before the parameter goes out of range
    date = rebuild_date(starting_date, days)
    return f'Проверьте параметр "{transcripts[column]}" датчика в течение {form_days(days)} (до {date})'


if __name__ == '__main__':
    app.run(debug=False)

dynamic_range: 20+
Mean MAE: 0.050668816317838934 (0.029301261524763655)
viewing_angle: 20+
Mean MAE: 0.001752161987473233 (0.0007557436149924564)
focal_length: 20+
Mean MAE: 0.019207824299281303 (0.012364545952161514)
temperature: 20+
Mean MAE: 0.003575177613431996 (0.0018862882703944174)
oscillation_frequency: 20+
Mean MAE: 0.002760267374997009 (0.0013473151531689831)
